# Supplementary Figure 1c: image processing and two-oligo quantification

This notebook is a cleaned copy of the original two-oligo RCA-FISH image-processing notebook. It retains the complete path from raw microscope TIFFs through Fiji stitching, LoG spot detection, Cellpose nuclear segmentation, spot-to-cell assignment, CSV summary construction, and the Supplementary Figure 1c plot.

**Repository behavior:** raw inputs and reference outputs are read from `../Data/`. When executed, intermediate stitching files, masks, and CSVs are written only to a temporary runtime directory that is deleted at the end. Figures are displayed in the notebook and are not exported. This notebook has not been re-executed after cleaning.

Source notebook: `code/Ian Anderson/Yang_image_processing_2oligo_quantification.ipynb`.


## Setup

In [ ]:
from IPython.display import display
import re
import json
import subprocess
import warnings
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy import ndimage as ndi
from scipy.ndimage import maximum_filter
from scipy.spatial import cKDTree
from skimage.segmentation import find_boundaries

warnings.filterwarnings("ignore")
print("Core imports OK")

try:
    from cellpose import models as cp_models
    from cellpose import io as cp_io
    cp_io.logger_setup()
    CELLPOSE_AVAILABLE = True
    print("Cellpose available")
except ImportError:
    CELLPOSE_AVAILABLE = False
    print("WARNING: Cellpose not installed. Run:  pip install cellpose")
    print("         Segmentation cells will be skipped.")


## Configuration

Edit this cell to configure all paths, channels, and analysis settings.

In [ ]:
from pathlib import Path
import os
import tempfile


def find_nanostamp_folder(start=Path.cwd()):
    start = Path(start).resolve()
    for base in (start, *start.parents):
        candidate = base / "Manuscripts" / "NanoSTAMP"
        if candidate.is_dir():
            return candidate
        if base.name == "NanoSTAMP" and (base / "Data").is_dir():
            return base
    raise FileNotFoundError("Could not locate Manuscripts/NanoSTAMP.")


NANOSTAMP_DIR = find_nanostamp_folder()
PUBLICATION_DATA_DIR = (
    NANOSTAMP_DIR / "Data" / "Supplementary_Figure_1c_2Oligo_RCA"
)
DATA_ROOT = PUBLICATION_DATA_DIR / "Raw_Images"
REFERENCE_OUTPUT_ROOT = PUBLICATION_DATA_DIR / "Processed_Data"

# All runtime intermediates are temporary and are removed in the final cell.
_runtime_cache = tempfile.TemporaryDirectory(prefix="nanostamp_supfig1c_")
OUTPUT_ROOT = Path(_runtime_cache.name)

# Only samples contributing to Supplementary Figure 1c are processed.
SELECTED_SAMPLE_FOLDERS = [
    "B10_2026-02-27_17-04-09.302384",
    "B8_2026-02-27_16-56-27.406809",
    "C11_2026-02-27_17-02-43.916718",
    "C3_1_2026-02-26_17-11-36.990927",
    "C5_2026-02-26_17-04-44.335246",
    "C7_2026-02-27_16-55-00.641006",
    "D10_2026-02-27_16-57-48.341383",
    "D11_2026-02-27_17-01-18.287206",
    "E10_2026-02-27_16-59-01.446545",
    "E11_2026-02-27_17-00-05.688559",
    "PBS_2026-02-26_16-59-28.287747",
    "PBS_2_2026-02-26_17-01-06.874313",
]

BARCODE_CHANNELS = {
    "barcode1": "Cy3-F46",
    "barcode2": "Cy5-F20",
}
NUCLEAR_CHANNEL = "DAPI"
CHANNEL_FILE_SUFFIX = {
    "DAPI": "Fluorescence_405_nm_Ex",
    "Cy3-F46": "Fluorescence_561_nm_Ex",
    "Cy5-F20": "Fluorescence_638_nm_Ex",
}

MIN_SPOTS_PER_BARCODE = 1

RUN_STITCHING = True
RUN_REGISTRATION = False
RUN_SPOT_DETECTION = True
RUN_SEGMENTATION = True
RUN_ASSIGNMENT = True
RUN_VISUALIZATION = True

# Set FIJI_PATH in the environment or edit the fallback for the local machine.
FIJI_PATH = Path(
    os.environ.get(
        "FIJI_PATH",
        "/Applications/Fiji.app/Contents/MacOS/ImageJ-macosx",
    )
)
TILE_OVERLAP_PCT = 10
STITCH_CHANNELS = ["DAPI", "Cy3-F46", "Cy5-F20"]

LOG_SIGMAS = [1.0, 2.0, 3.0, 4.0]
PEAK_WIDTH = 11
NMS_MIN_DISTANCE = 5
THRESHOLD_PEAKS_BY_BARCODE = {
    "barcode1": 1000,
    "barcode2": 2000,
}
THRESHOLD_PEAKS = 500
MAX_FILTER_WIDTH = 3

CELLPOSE_DIAMETER = 25
CELLPOSE_CHANNELS = [0, 0]
# Use a GPU when available; set CELLPOSE_GPU=0 or CELLPOSE_GPU=1 to override.
_cellpose_gpu_setting = os.environ.get("CELLPOSE_GPU", "auto").strip().lower()
if _cellpose_gpu_setting == "auto":
    try:
        import torch
        CELLPOSE_GPU = bool(torch.cuda.is_available() or torch.backends.mps.is_available())
    except Exception:
        CELLPOSE_GPU = False
else:
    CELLPOSE_GPU = _cellpose_gpu_setting in {"1", "true", "yes", "on"}
MIN_NUCLEUS_AREA_PX = 100
MAX_NUCLEUS_AREA_PX = 5000
EXPAND_MASK_PX = 10

EXCLUDE_FOVS_BY_SAMPLE = {}
N_VIS_SAMPLES = 1
N_VIS_FOVS = 1

print("Raw images:", DATA_ROOT)
print("Reference processed data:", REFERENCE_OUTPUT_ROOT)
print("Temporary runtime directory:", OUTPUT_ROOT)


## File Discovery / Box Drive Checks

Validates that DATA_ROOT and selected sample folders exist and that image files can actually be opened (Box Drive online-only files will fail here).

In [ ]:
from pathlib import Path
import tifffile


def get_real_tiffs(folder):
    folder = Path(folder)
    return sorted(
        path
        for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in {".tif", ".tiff"}
        and not path.name.startswith(("._", ".DS_Store"))
        and path.stat().st_size > 100_000
    )


def get_fake_tiffs(folder):
    folder = Path(folder)
    return sorted(
        path
        for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in {".tif", ".tiff"}
        and path.name.startswith("._")
    )


if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Raw-image folder not found: {DATA_ROOT}")
if not FIJI_PATH.exists():
    print(f"Fiji not found at {FIJI_PATH}; set FIJI_PATH before running stitching.")

valid_samples = []
sample_folders = []
SAMPLE_INFO = {}

for folder_name in SELECTED_SAMPLE_FOLDERS:
    sample_folder = DATA_ROOT / folder_name
    img_dir = sample_folder / "0"
    if not img_dir.is_dir():
        raise FileNotFoundError(f"Missing microscope image folder: {img_dir}")

    tiff_files = get_real_tiffs(img_dir)
    if not tiff_files:
        raise FileNotFoundError(f"No real TIFF files found in: {img_dir}")

    test_file = tiff_files[0]
    test_image = tifffile.imread(test_file)
    tile_prefix = test_file.name.split("_")[0]

    valid_samples.append(sample_folder)
    sample_folders.append(sample_folder)
    SAMPLE_INFO[sample_folder] = {
        "sample_name": sample_folder.name,
        "img_dir": img_dir,
        "sample_output": OUTPUT_ROOT / sample_folder.name,
        "tile_prefix": tile_prefix,
        "tiff_files": tiff_files,
        "fake_tiff_files": get_fake_tiffs(img_dir),
        "n_real_tiffs": len(tiff_files),
        "test_file": test_file,
        "test_shape": test_image.shape,
        "dtype": test_image.dtype,
    }

SAMPLE_FOLDER = valid_samples[0]
SAMPLE_FOLDER_NAME = SAMPLE_FOLDER.name
IMG_DIR = SAMPLE_INFO[SAMPLE_FOLDER]["img_dir"]
SAMPLE_OUTPUT = SAMPLE_INFO[SAMPLE_FOLDER]["sample_output"]
TILE_PREFIX = SAMPLE_INFO[SAMPLE_FOLDER]["tile_prefix"]
tiff_files = SAMPLE_INFO[SAMPLE_FOLDER]["tiff_files"]
fake_tiff_files = SAMPLE_INFO[SAMPLE_FOLDER]["fake_tiff_files"]

validation_table = pd.DataFrame(
    {
        "sample": [sample.name for sample in valid_samples],
        "real_tiffs": [SAMPLE_INFO[sample]["n_real_tiffs"] for sample in valid_samples],
        "test_shape": [SAMPLE_INFO[sample]["test_shape"] for sample in valid_samples],
        "dtype": [str(SAMPLE_INFO[sample]["dtype"]) for sample in valid_samples],
    }
)
display(validation_table)


## Stitching

Uses Fiji Grid/Collection stitching to assemble per-FOV tiles into a full-canvas image for each channel. Controlled by `RUN_STITCHING` and `FIJI_PATH`.

> **Note:** Stitching is for QC overview visualization. Spot detection and segmentation run on individual FOV tiles and do not require stitched images.

In [ ]:
from pathlib import Path
import re
import subprocess
import pandas as pd


def _real_tiff_files(img_dir: Path):
    # Only real microscope TIFFs, not Mac/Box metadata files like ._B8...
    img_dir = Path(img_dir)
    return sorted([
        p for p in img_dir.glob("*.tif*")
        if p.is_file()
        and not p.name.startswith("._")
        and not p.name.startswith(".DS_Store")
        and p.stat().st_size > 100_000
    ])


def _fake_tiff_files(img_dir: Path):
    img_dir = Path(img_dir)
    return sorted([
        p for p in img_dir.glob("*.tif*")
        if p.is_file()
        and p.name.startswith("._")
    ])


def _detect_tile_prefix(img_dir: Path) -> str:
    # Example: B8_0_0_Fluorescence_405_nm_Ex.tiff -> B8
    files = _real_tiff_files(img_dir)

    if not files:
        raise ValueError(f"No real TIFF files found in {img_dir}")

    print("  Prefix detection using:", files[0].name)

    for f in files:
        m = re.match(r'^(.+?)_\d+_\d+_', f.name)
        if m:
            prefix = m.group(1)
            if not prefix.startswith("."):
                return prefix

    raise ValueError(f"Cannot detect tile prefix in {img_dir}")


def _get_grid_size(img_dir: Path):
    # Return (n_cols, n_rows) from coordinates.csv if present.
    img_dir = Path(img_dir)
    csv = img_dir / "coordinates.csv"

    if csv.exists():
        df = pd.read_csv(csv)
        n_x = round(df["x (mm)"].nunique())
        n_y = round(df["y (mm)"].nunique())
        return int(n_x), int(n_y)

    # Fallback: infer square-ish grid from unique tile indices.
    files = _real_tiff_files(img_dir)
    tile_nums = set()

    for f in files:
        m = re.match(r'^(.+?)_(\d+)_(\d+)_', f.name)
        if m:
            tile_nums.add(int(m.group(2)))

    if tile_nums:
        n_tiles = len(tile_nums)
        side = int(round(n_tiles ** 0.5))
        if side * side == n_tiles:
            return side, side

    return None, None


def _make_fiji_macro(img_dir, out_dir, prefix, ch_suffix, n_x, n_y, overlap_pct):
    d = str(img_dir).replace("\\", "/")
    od = str(out_dir).replace("\\", "/")
    safe = ch_suffix.replace(" ", "_")

    return (
        f'dir = "{d}/";\n'
        f'outdir = "{od}/";\n'
        f'run("Grid/Collection stitching",\n'
        f'    "type=[Grid: row-by-row] " +\n'
        f'    "order=[Right & Down                ] " +\n'
        f'    "grid_size_x={n_x} " +\n'
        f'    "grid_size_y={n_y} " +\n'
        f'    "tile_overlap={overlap_pct} " +\n'
        f'    "first_file_index_i=0 " +\n'
        f'    "directory=[" + dir + "] " +\n'
        f'    "file_names={prefix}_{{i}}_0_{ch_suffix}.tiff " +\n'
        f'    "output_textfile_name=TileConfiguration_{safe}.txt " +\n'
        f'    "fusion_method=[Linear Blending] " +\n'
        f'    "regression_threshold=0.30 " +\n'
        f'    "max/avg_displacement_threshold=2.50 " +\n'
        f'    "absolute_displacement_threshold=3.50 " +\n'
        f'    "compute_overlap " +\n'
        f'    "computation_parameters=[Save memory (but be slower)] " +\n'
        f'    "image_output=[Fuse and display]");\n'
        f'saveAs("Tiff", outdir + "stitched_{safe}.tif");\n'
        f'close();\n'
        f'run("Quit");\n'
    )


print("Stitching helper functions loaded.")
print("Ready to run stitching over all folders in valid_samples.")

In [ ]:
if RUN_STITCHING:
    if not FIJI_PATH.exists():
        print(f"WARNING: Fiji not found at {FIJI_PATH}")
        print("Set FIJI_PATH in the Configuration cell to your Fiji executable.")
        print("Stitching SKIPPED.")

    else:
        print(f"Fiji: {FIJI_PATH}")

        for sf in valid_samples:
            sf = Path(sf)
            sample_name = sf.name
            img_dir = sf / "0"
            stitch_out = OUTPUT_ROOT / sample_name / "stitched"
            stitch_out.mkdir(parents=True, exist_ok=True)

            print("\n" + "=" * 80)
            print(f"STITCHING SAMPLE: {sample_name}")
            print(f"IMG_DIR: {img_dir}")
            print(f"OUTPUT: {stitch_out}")
            print("=" * 80)

            try:
                real_files = _real_tiff_files(img_dir)
                fake_files = _fake_tiff_files(img_dir)

                print(f"  Real TIFFs used: {len(real_files)}")
                print(f"  Fake ._ TIFFs ignored: {len(fake_files)}")

                if not real_files:
                    print(f"  SKIP: no real TIFF files in {img_dir}")
                    continue

                prefix = _detect_tile_prefix(img_dir)
                n_x, n_y = _get_grid_size(img_dir)

            except Exception as exc:
                print(f"  SKIP: setup error for {sample_name}: {exc}")
                continue

            if n_x is None or n_y is None:
                print(f"  SKIP: cannot determine grid size for {sample_name}")
                continue

            if prefix.startswith("."):
                raise RuntimeError(f"BAD PREFIX DETECTED for {sample_name}: {prefix}")

            print(f"  Prefix: {prefix}")
            print(f"  Grid: {n_x} x {n_y}")

            for ch_name in STITCH_CHANNELS:
                ch_suffix = CHANNEL_FILE_SUFFIX.get(ch_name)

                if ch_suffix is None:
                    print(f"  [{ch_name}] No file suffix defined. Skipping.")
                    continue

                safe = ch_suffix.replace(" ", "_")
                out_tif = stitch_out / f"stitched_{safe}.tif"

                # Remove old tiny/bad output if it exists
                if out_tif.exists() and out_tif.stat().st_size < 100_000:
                    print(f"  [{ch_name}] Removing bad old output: {out_tif.name}")
                    out_tif.unlink()

                if out_tif.exists() and out_tif.stat().st_size > 100_000:
                    print(f"  [{ch_name}] Already stitched: {out_tif.name} size={out_tif.stat().st_size}")
                    continue

                macro_text = _make_fiji_macro(
                    img_dir=img_dir,
                    out_dir=stitch_out,
                    prefix=prefix,
                    ch_suffix=ch_suffix,
                    n_x=n_x,
                    n_y=n_y,
                    overlap_pct=TILE_OVERLAP_PCT
                )

                macro_path = stitch_out / f"stitch_{ch_name}.ijm"
                macro_path.write_text(macro_text, encoding="utf-8")

                print(f"  [{ch_name}] Running Fiji...", flush=True)

                result = subprocess.run(
                    [str(FIJI_PATH), "--headless", "--console", "-macro", str(macro_path)],
                    capture_output=True,
                    text=True,
                    timeout=900
                )

                if out_tif.exists() and out_tif.stat().st_size > 100_000:
                    print(f"  [{ch_name}] DONE -> {out_tif.name} size={out_tif.stat().st_size}")
                else:
                    print(f"  [{ch_name}] WARN: output not created correctly. Fiji exit code {result.returncode}")

                    if result.stderr:
                        print("  stderr:")
                        print(result.stderr[:1500])

                    if result.stdout:
                        print("  stdout:")
                        print(result.stdout[:1500])

        print("\n" + "=" * 80)
        print("Stitching step complete.")
        print("=" * 80)

        print("\nChecking stitched outputs:")
        for sf in valid_samples:
            sf = Path(sf)
            stitch_out = OUTPUT_ROOT / sf.name / "stitched"
            stitched_tifs = sorted(list(stitch_out.glob("*.tif")) + list(stitch_out.glob("*.tiff")))

            print(f"\n{sf.name}: {len(stitched_tifs)} stitched TIFFs")
            for p in stitched_tifs:
                print(f"  {p.name} | size={p.stat().st_size}")

else:
    print("Stitching skipped because RUN_STITCHING=False.")

# Display one representative DAPI tile beside its Fiji-stitched field.
if RUN_STITCHING and FIJI_PATH.exists() and valid_samples:
    representative = valid_samples[0]
    representative_info = SAMPLE_INFO[representative]
    dapi_suffix = CHANNEL_FILE_SUFFIX["DAPI"]
    raw_tile_path = next(
        path for path in representative_info["tiff_files"]
        if dapi_suffix in path.name
    )
    raw_tile = tifffile.imread(raw_tile_path)
    if raw_tile.ndim > 2:
        raw_tile = raw_tile.reshape((-1, raw_tile.shape[-2], raw_tile.shape[-1]))[0]
    stitched_path = (
        OUTPUT_ROOT
        / representative.name
        / "stitched"
        / "stitched_Fluorescence_405_nm_Ex.tif"
    )
    if stitched_path.exists():
        stitched_image = tifffile.imread(stitched_path)
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(raw_tile, cmap="gray")
        axes[0].set_title("Representative raw DAPI tile")
        axes[1].imshow(stitched_image, cmap="gray")
        axes[1].set_title("Fiji Grid/Collection stitching")
        for axis in axes:
            axis.axis("off")
        fig.tight_layout()
        plt.show()
        plt.close(fig)


## Spot Detection

Detects RCA-FISH spots in each FOV tile using multi-scale Laplacian of Gaussian (LoG). Runs independently for barcode1 (Cy3-F46) and barcode2 (Cy5-F20).

**Key parameters to tune:** `LOG_SIGMAS`, `THRESHOLD_PEAKS`, `NMS_MIN_DISTANCE`

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import tifffile
from scipy import ndimage as ndi
from scipy.ndimage import maximum_filter
from scipy.spatial import cKDTree

# ── TIFF helpers: ignore fake ._ files ──────────────────────────────────────

def _real_tiff_files(img_dir: Path):
    img_dir = Path(img_dir)
    return sorted([
        p for p in img_dir.glob("*.tif*")
        if p.is_file()
        and not p.name.startswith("._")
        and p.stat().st_size > 100_000
    ])


def _detect_tile_prefix_for_spots(img_dir: Path) -> str:
    files = _real_tiff_files(img_dir)

    if not files:
        raise ValueError(f"No real TIFF files found in {img_dir}")

    print("  Spot prefix detection using:", files[0].name)

    for f in files:
        m = re.match(r'^(.+?)_\d+_\d+_', f.name)
        if m:
            prefix = m.group(1)
            if not prefix.startswith("."):
                return prefix

    raise ValueError(f"Cannot detect tile prefix in {img_dir}")


# ── I/O helpers ─────────────────────────────────────────────────────────────

def read_yx(path: Path) -> np.ndarray:
    path = Path(path)

    if path.name.startswith("._"):
        raise ValueError(f"Refusing to read fake Mac metadata file: {path}")

    if path.stat().st_size < 100_000:
        raise ValueError(f"Refusing to read tiny/non-image TIFF file: {path} size={path.stat().st_size}")

    arr = tifffile.imread(str(path))
    arr = np.squeeze(arr)

    if arr.ndim > 2:
        arr = arr.reshape((-1, arr.shape[-2], arr.shape[-1]))[0]

    return arr.astype(np.float32, copy=False)


def _remove_hot_pixels(img: np.ndarray, width: int = 3) -> np.ndarray:
    local_max = maximum_filter(img, size=width)
    hot = img > np.percentile(img, 99.99)
    out = img.copy()
    out[hot] = local_max[hot]
    return out


def detect_spots_log(
    img: np.ndarray,
    log_sigmas=None,
    peak_width: int = 11,
    nms_min_distance: int = 5,
    threshold: float = 500,
) -> np.ndarray:
    if log_sigmas is None:
        log_sigmas = [1.0, 2.0, 3.0]

    img = _remove_hot_pixels(img)

    log_stack = []

    for sigma in log_sigmas:
        resp = -ndi.gaussian_laplace(img.astype(np.float64), sigma=sigma) * (sigma ** 2)
        log_stack.append(resp.astype(np.float32))

    log_max = np.max(np.stack(log_stack, axis=0), axis=0)

    local_max = maximum_filter(log_max, size=peak_width)
    candidates = np.where((log_max == local_max) & (log_max > threshold))
    ys, xs = candidates[0], candidates[1]

    if len(ys) == 0:
        return np.zeros((0, 4), dtype=np.float32)

    intensities = img[ys, xs]
    scores = log_max[ys, xs]

    coords = np.column_stack([ys, xs]).astype(np.float64)
    order = np.argsort(-scores)
    tree = cKDTree(coords)

    taken = set()
    keep = []

    for idx in order:
        if idx in taken:
            continue

        keep.append(idx)
        neighbours = tree.query_ball_point(coords[idx], r=nms_min_distance)
        taken.update(neighbours)

    keep = np.asarray(keep, dtype=int)

    return np.column_stack(
        [ys[keep], xs[keep], intensities[keep], scores[keep]]
    ).astype(np.float32)


def _get_fov_ids(img_dir: Path, nuclear_suffix: str) -> list:
    img_dir = Path(img_dir)

    csv = img_dir / "coordinates.csv"

    if csv.exists():
        df = pd.read_csv(csv)

        if "fov" in df.columns:
            return sorted(df["fov"].astype(int).unique().tolist())

    pattern = re.compile(
        r'^.+?_(\d+)_\d+_' + re.escape(nuclear_suffix) + r'\.tiff$'
    )

    ids = []

    for f in _real_tiff_files(img_dir):
        m = pattern.match(f.name)
        if m:
            ids.append(int(m.group(1)))

    return sorted(set(ids))


# ── Main spot detection loop ────────────────────────────────────────────────

if RUN_SPOT_DETECTION:
    all_spot_records = []

    for sf in valid_samples:
        sample_name = sf.name
        img_dir = sf / "0"
        spot_out = OUTPUT_ROOT / sample_name / "spot_detection"
        spot_out.mkdir(parents=True, exist_ok=True)

        real_files = _real_tiff_files(img_dir)
        fake_files = sorted([
            p for p in img_dir.glob("*.tif*")
            if p.is_file() and p.name.startswith("._")
        ])

        print(f"\nSample: {sample_name}")
        print(f"  Real TIFFs available: {len(real_files)}")
        print(f"  Fake ._ TIFFs ignored: {len(fake_files)}")

        try:
            tile_prefix = _detect_tile_prefix_for_spots(img_dir)
        except Exception as exc:
            print(f"Cannot detect prefix for {sample_name}: {exc}. Skipping.")
            continue

        if tile_prefix.startswith("."):
            raise RuntimeError(f"BAD tile_prefix detected: {tile_prefix}")

        nuc_suffix = CHANNEL_FILE_SUFFIX[NUCLEAR_CHANNEL]
        fov_ids = _get_fov_ids(img_dir, nuc_suffix)

        print(f"  Tile prefix: {tile_prefix}")
        print(f"  Nuclear suffix: {nuc_suffix}")
        print(f"  FOVs detected: {len(fov_ids)}")
        print(f"\nSpot detection: {sample_name}  ({len(fov_ids)} FOVs)")

        sample_records = []

        for fov_id in fov_ids:
            for bc_key, ch_name in BARCODE_CHANNELS.items():
                ch_suffix = CHANNEL_FILE_SUFFIX.get(ch_name)

                if ch_suffix is None:
                    continue

                img_path = img_dir / f"{tile_prefix}_{fov_id}_0_{ch_suffix}.tiff"

                if img_path.name.startswith("._"):
                    print(f"  WARN: skipping fake file {img_path.name}")
                    continue

                if not img_path.exists():
                    print(f"  WARN: missing {img_path.name}")
                    continue

                if img_path.stat().st_size < 100_000:
                    print(f"  WARN: skipping tiny/bad file {img_path.name} size={img_path.stat().st_size}")
                    continue

                img = read_yx(img_path)

                threshold = THRESHOLD_PEAKS_BY_BARCODE.get(bc_key, THRESHOLD_PEAKS)

                spots = detect_spots_log(
                    img,
                    log_sigmas=LOG_SIGMAS,
                    peak_width=PEAK_WIDTH,
                    nms_min_distance=NMS_MIN_DISTANCE,
                    threshold=threshold,
                )

                for row in spots:
                    sample_records.append({
                        "sample": sample_name,
                        "fov": fov_id,
                        "barcode": bc_key,
                        "channel": ch_name,
                        "y": int(row[0]),
                        "x": int(row[1]),
                        "intensity": float(row[2]),
                        "log_score": float(row[3]),
                    })

            if fov_id % 5 == 0 or fov_id == fov_ids[-1]:
                b1 = sum(
                    1 for r in sample_records
                    if r["fov"] == fov_id and r["barcode"] == "barcode1"
                )
                b2 = sum(
                    1 for r in sample_records
                    if r["fov"] == fov_id and r["barcode"] == "barcode2"
                )
                print(f"  FOV {fov_id:3d}: barcode1={b1:4d}  barcode2={b2:4d}")

        all_spot_records.extend(sample_records)

        if sample_records:
            csv_path = spot_out / "spots_all_barcodes.csv"
            pd.DataFrame(sample_records).to_csv(csv_path, index=False)
            print(f"  Saved: {csv_path}")

            for bc_key in BARCODE_CHANNELS:
                n = sum(1 for r in sample_records if r["barcode"] == bc_key)
                print(f"  {bc_key}: {n} total spots")
        else:
            print("  No spots detected/saved for this sample.")

    spots_df = pd.DataFrame(all_spot_records)
    print(f"\nSpot detection complete. Total spots: {len(spots_df)}")

else:
    print("Spot detection skipped (RUN_SPOT_DETECTION=False). Loading existing results.")

    spots_df = pd.DataFrame()

    for sf in valid_samples:
        csv_path = OUTPUT_ROOT / sf.name / "spot_detection" / "spots_all_barcodes.csv"

        if csv_path.exists():
            spots_df = pd.concat(
                [spots_df, pd.read_csv(csv_path)],
                ignore_index=True
            )

    print(f"Loaded {len(spots_df)} spot records.")

# Always enforce the current thresholds here too. This matters when loading old
# spot CSVs that were originally detected with different threshold values.
if not spots_df.empty and {"barcode", "log_score"}.issubset(spots_df.columns):
    n_before_threshold_filter = len(spots_df)
    threshold_by_row = spots_df["barcode"].map(THRESHOLD_PEAKS_BY_BARCODE).fillna(THRESHOLD_PEAKS)
    spots_df = spots_df[spots_df["log_score"] >= threshold_by_row].copy()
    print(
        "Applied per-barcode log_score thresholds "
        f"{THRESHOLD_PEAKS_BY_BARCODE}: kept {len(spots_df)} / "
        f"{n_before_threshold_filter} spots"
    )




## Nucleus/Cell Segmentation

Runs Cellpose nuclear segmentation on the DAPI channel for each FOV tile. Masks are saved as `.npy` and `.tif` files under `OUTPUT_ROOT/<sample>/segmentation/`.

**Key parameters:** `CELLPOSE_DIAMETER`, `MIN_NUCLEUS_AREA_PX`, `MAX_NUCLEUS_AREA_PX`

### Cellpose model requirement

Install `cellpose` before execution and place the Cellpose-SAM `cpsam` model at `~/.cellpose/models/cpsam`. The segmentation cell below checks that file explicitly and records the exact model path, diameter, channel, GPU, and area-filter settings used in the original analysis.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import tifffile

# ── Check Cellpose availability ─────────────────────────────────────────────

try:
    from cellpose import models as cp_models
    CELLPOSE_AVAILABLE = True
except Exception as exc:
    CELLPOSE_AVAILABLE = False
    print("Cellpose import failed:", exc)


# ── Helpers: ignore fake ._ TIFFs ───────────────────────────────────────────

def _real_tiff_files_for_seg(img_dir: Path):
    img_dir = Path(img_dir)
    return sorted([
        p for p in img_dir.glob("*.tif*")
        if p.is_file()
        and not p.name.startswith("._")
        and not p.name.startswith(".DS_Store")
        and p.stat().st_size > 100_000
    ])


def _detect_tile_prefix_for_seg(img_dir: Path) -> str:
    files = _real_tiff_files_for_seg(img_dir)

    if not files:
        raise ValueError(f"No real TIFF files found in {img_dir}")

    print("  Segmentation prefix detection using:", files[0].name)

    for f in files:
        m = re.match(r'^(.+?)_\d+_\d+_', f.name)
        if m:
            prefix = m.group(1)
            if not prefix.startswith("."):
                return prefix

    raise ValueError(f"Cannot detect tile prefix in {img_dir}")


def _get_fov_ids_for_seg(img_dir: Path, nuclear_suffix: str) -> list:
    img_dir = Path(img_dir)

    csv = img_dir / "coordinates.csv"

    if csv.exists():
        df = pd.read_csv(csv)

        if "fov" in df.columns:
            return sorted(df["fov"].astype(int).unique().tolist())

    pattern = re.compile(
        r'^.+?_(\d+)_\d+_' + re.escape(nuclear_suffix) + r'\.tiff$'
    )

    ids = []

    for f in _real_tiff_files_for_seg(img_dir):
        m = pattern.match(f.name)
        if m:
            ids.append(int(m.group(1)))

    return sorted(set(ids))


def read_yx(path: Path) -> np.ndarray:
    path = Path(path)

    if path.name.startswith("._"):
        raise ValueError(f"Refusing to read fake ._ file: {path}")

    if path.stat().st_size < 100_000:
        raise ValueError(f"Refusing to read tiny/bad TIFF: {path} size={path.stat().st_size}")

    arr = tifffile.imread(str(path))
    arr = np.squeeze(arr)

    if arr.ndim > 2:
        arr = arr.reshape((-1, arr.shape[-2], arr.shape[-1]))[0]

    return arr.astype(np.float32, copy=False)


# ── Main segmentation over ALL valid samples ─────────────────────────────────

if RUN_SEGMENTATION:
    if not CELLPOSE_AVAILABLE:
        print("ERROR: Cellpose not installed. Segmentation skipped.")
        mask_results = {}

    else:
        model_path = Path.home() / ".cellpose" / "models" / "cpsam"

        if not model_path.exists() or model_path.stat().st_size < 1_000_000:
            raise FileNotFoundError(
                f"Cellpose model not found or too small at:\n{model_path}\n"
                "Run the wget model-download cell again first."
            )

        print("Using Cellpose model:", model_path)
        print("Model size:", model_path.stat().st_size)
        print("GPU setting:", CELLPOSE_GPU)

        cellpose_model = cp_models.CellposeModel(
            gpu=CELLPOSE_GPU,
            pretrained_model=str(model_path)
        )

        mask_results = {}

        for sf in valid_samples:
            sf = Path(sf)
            sample_name = sf.name
            img_dir = sf / "0"
            seg_out = OUTPUT_ROOT / sample_name / "segmentation"
            seg_out.mkdir(parents=True, exist_ok=True)

            print("\n" + "=" * 80)
            print(f"SEGMENTING SAMPLE: {sample_name}")
            print(f"IMG_DIR: {img_dir}")
            print(f"OUTPUT: {seg_out}")
            print("=" * 80)

            real_files = _real_tiff_files_for_seg(img_dir)
            fake_files = sorted([
                p for p in img_dir.glob("*.tif*")
                if p.is_file() and p.name.startswith("._")
            ])

            print(f"  Real TIFFs available: {len(real_files)}")
            print(f"  Fake ._ TIFFs ignored: {len(fake_files)}")

            if not real_files:
                print(f"  SKIP: no real TIFFs in {img_dir}")
                continue

            try:
                tile_prefix = _detect_tile_prefix_for_seg(img_dir)
            except Exception as exc:
                print(f"  SKIP: cannot detect prefix for {sample_name}: {exc}")
                continue

            if tile_prefix.startswith("."):
                raise RuntimeError(f"BAD tile_prefix detected: {tile_prefix}")

            nuc_suffix = CHANNEL_FILE_SUFFIX[NUCLEAR_CHANNEL]
            fov_ids = _get_fov_ids_for_seg(img_dir, nuc_suffix)

            print(f"  Tile prefix: {tile_prefix}")
            print(f"  Nuclear suffix: {nuc_suffix}")
            print(f"  FOVs detected: {len(fov_ids)}")

            if not fov_ids:
                print(f"  SKIP: no FOVs detected for {sample_name}")
                continue

            for fov_id in fov_ids:
                npy_path = seg_out / f"mask_fov{fov_id:03d}.npy"
                tif_path = seg_out / f"mask_fov{fov_id:03d}.tif"

                # Load cached mask if it already exists
                if npy_path.exists():
                    masks = np.load(str(npy_path))
                    mask_results[(sample_name, fov_id)] = masks

                    if fov_id % 5 == 0 or fov_id == fov_ids[-1]:
                        print(f"  FOV {fov_id:3d}: loaded cached mask, {int(masks.max())} nuclei")

                    continue

                dapi_path = img_dir / f"{tile_prefix}_{fov_id}_0_{nuc_suffix}.tiff"

                if not dapi_path.exists():
                    print(f"  WARN: no DAPI for FOV {fov_id}: {dapi_path.name}")
                    continue

                if dapi_path.name.startswith("._"):
                    print(f"  WARN: skipping fake file {dapi_path.name}")
                    continue

                if dapi_path.stat().st_size < 100_000:
                    print(f"  WARN: skipping tiny/bad DAPI {dapi_path.name} size={dapi_path.stat().st_size}")
                    continue

                dapi_img = read_yx(dapi_path)

                masks_raw, _, _ = cellpose_model.eval(
                    dapi_img,
                    channels=CELLPOSE_CHANNELS,
                    diameter=CELLPOSE_DIAMETER
                )

                # ── Size filter ──────────────────────────────────────────────
                labels, counts = np.unique(masks_raw, return_counts=True)

                fg = labels > 0
                labels = labels[fg]
                counts = counts[fg]

                valid_ids = labels[
                    (counts >= MIN_NUCLEUS_AREA_PX) &
                    (counts <= MAX_NUCLEUS_AREA_PX)
                ]

                filtered = masks_raw.copy()
                filtered[~np.isin(filtered, valid_ids)] = 0

                # ── Relabel consecutively ───────────────────────────────────
                old_ids = np.unique(filtered)
                old_ids = old_ids[old_ids > 0]

                if len(old_ids) > 0:
                    rmap = np.zeros(int(filtered.max()) + 1, dtype=np.int32)

                    for new_id, old_id in enumerate(old_ids, start=1):
                        rmap[old_id] = new_id

                    filtered = rmap[filtered]
                else:
                    filtered = np.zeros_like(filtered, dtype=np.int32)

                filtered = filtered.astype(np.int32)

                # ── Save masks ───────────────────────────────────────────────
                np.save(str(npy_path), filtered)
                tifffile.imwrite(str(tif_path), filtered)

                mask_results[(sample_name, fov_id)] = filtered

                if fov_id % 5 == 0 or fov_id == fov_ids[-1]:
                    print(f"  FOV {fov_id:3d}: {int(filtered.max())} nuclei")

            n_total = sum(
                int(v.max())
                for (s, f), v in mask_results.items()
                if s == sample_name
            )

            print(f"\n  Total nuclei in {sample_name}: {n_total}")

        print("\nSegmentation complete.")
        print(f"Total segmented FOVs across all samples: {len(mask_results)}")


else:
    print("Segmentation skipped because RUN_SEGMENTATION=False. Loading existing masks.")

    mask_results = {}

    for sf in valid_samples:
        sf = Path(sf)
        seg_dir = OUTPUT_ROOT / sf.name / "segmentation"

        if not seg_dir.exists():
            print(f"  No segmentation folder for {sf.name}")
            continue

        for npy_file in sorted(seg_dir.glob("mask_fov*.npy")):
            m = re.match(r'mask_fov(\d+)\.npy', npy_file.name)

            if m:
                fov_id = int(m.group(1))
                mask_results[(sf.name, fov_id)] = np.load(str(npy_file))

    print(f"Loaded masks for {len(mask_results)} FOV(s).")

## Spot Assignment to Cells

For each cell, counts barcode1 and barcode2 spots that fall within the nucleus mask (expanded by `EXPAND_MASK_PX` pixels). Classifies each cell as:
- `barcode1_only` – only barcode1 present
- `barcode2_only` – only barcode2 present
- `both_barcodes` – both present
- `no_barcode` – neither present

Threshold: `MIN_SPOTS_PER_BARCODE` spots required for a barcode to be 'present'.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

if RUN_ASSIGNMENT:
    if len(spots_df) == 0:
        print("No spots available. Run spot detection first.")
        cell_df = pd.DataFrame()

    elif not mask_results:
        print("No segmentation masks. Run segmentation first.")
        cell_df = pd.DataFrame()

    else:
        per_cell_rows = []
        barcode_keys = list(BARCODE_CHANNELS.keys())

        # ── Loop over EVERY valid sample ────────────────────────────────────
        for sf in valid_samples:
            sf = Path(sf)
            sample_name = sf.name

            sample_spots = spots_df[spots_df["sample"] == sample_name].copy()

            if not sample_spots.empty:
                sample_spots["fov"] = sample_spots["fov"].astype(int)

            excluded_fovs = set(EXCLUDE_FOVS_BY_SAMPLE.get(sample_name, []))
            if excluded_fovs and not sample_spots.empty:
                sample_spots = sample_spots[~sample_spots["fov"].isin(excluded_fovs)].copy()

            sample_fovs = sorted([
                int(f)
                for (s, f) in mask_results.keys()
                if s == sample_name and int(f) not in excluded_fovs
            ])

            if excluded_fovs:
                print(f"  Excluding QC-failed FOVs for {sample_name}: {sorted(excluded_fovs)}")

            print("\n" + "=" * 80)
            print(f"ASSIGNING SPOTS TO CELLS: {sample_name}")
            print(f"  FOVs with masks: {len(sample_fovs)}")
            print(f"  Spots in sample: {len(sample_spots)}")
            print("=" * 80)

            if len(sample_fovs) == 0:
                print(f"  SKIP: no masks found for {sample_name}")
                continue

            for fov_id in sample_fovs:
                masks = mask_results[(sample_name, fov_id)].astype(np.int32)
                n_cells = int(masks.max())

                if n_cells == 0:
                    continue

                # ── Expand nucleus masks outward ────────────────────────────
                if EXPAND_MASK_PX > 0:
                    nuc_binary = masks > 0

                    dist, nearest = ndi.distance_transform_edt(
                        ~nuc_binary,
                        return_indices=True
                    )

                    expanded = masks.copy()
                    zone = (dist <= EXPAND_MASK_PX) & (~nuc_binary)

                    expanded[zone] = masks[
                        nearest[0][zone],
                        nearest[1][zone]
                    ]

                else:
                    expanded = masks

                # ── Count spots per cell ────────────────────────────────────
                fov_spots = sample_spots[sample_spots["fov"] == int(fov_id)]

                cell_counts = {}

                for _, spot in fov_spots.iterrows():
                    y = int(spot["y"])
                    x = int(spot["x"])

                    if not (0 <= y < expanded.shape[0] and 0 <= x < expanded.shape[1]):
                        continue

                    cell_id = int(expanded[y, x])

                    if cell_id == 0:
                        continue

                    bc = spot["barcode"]

                    counts_for_cell = cell_counts.setdefault(
                        cell_id,
                        {k: 0 for k in barcode_keys}
                    )

                    counts_for_cell[bc] = counts_for_cell.get(bc, 0) + 1

                # ── Classify ALL cells, including zero-spot cells ────────────
                for cell_id in range(1, n_cells + 1):
                    counts = cell_counts.get(
                        cell_id,
                        {k: 0 for k in barcode_keys}
                    )

                    b1 = int(counts.get("barcode1", 0))
                    b2 = int(counts.get("barcode2", 0))

                    b1_pres = b1 >= MIN_SPOTS_PER_BARCODE
                    b2_pres = b2 >= MIN_SPOTS_PER_BARCODE

                    if b1_pres and not b2_pres:
                        clf = "barcode1_only"
                    elif b2_pres and not b1_pres:
                        clf = "barcode2_only"
                    elif b1_pres and b2_pres:
                        clf = "both_barcodes"
                    else:
                        clf = "no_barcode"

                    per_cell_rows.append({
                        "sample": sample_name,
                        "fov": int(fov_id),
                        "cell_id": int(cell_id),
                        "barcode1_spot_count": b1,
                        "barcode2_spot_count": b2,
                        "barcode1_present": bool(b1_pres),
                        "barcode2_present": bool(b2_pres),
                        "classification": clf,
                    })

                if fov_id % 5 == 0 or fov_id == sample_fovs[-1]:
                    print(
                        f"  FOV {fov_id:3d}: "
                        f"{n_cells} cells, "
                        f"{len(fov_spots)} spots"
                    )

        cell_df = pd.DataFrame(per_cell_rows)

        if not cell_df.empty:
            # ── Save one CSV per sample ─────────────────────────────────────
            for sname in cell_df["sample"].unique():
                csv_path = OUTPUT_ROOT / sname / "per_cell_2oligo_barcode_assignment.csv"
                cell_df[cell_df["sample"] == sname].to_csv(csv_path, index=False)
                print(f"\nSaved: {csv_path}")

            # ── Save combined CSV across all samples ────────────────────────
            combined_csv = OUTPUT_ROOT / "combined_per_cell_2oligo_barcode_assignment.csv"
            cell_df.to_csv(combined_csv, index=False)
            print(f"Saved combined table: {combined_csv}")

            print(f"\nAssignment complete. Total cells across all samples: {len(cell_df)}")
            print()
            print("Overall classification counts:")
            print(cell_df["classification"].value_counts().to_string())

            # ── FINAL TWO NUMBERS, COMBINED ACROSS ALL SAMPLES ──────────────
            one_barcode = int(
                cell_df["classification"].isin(
                    ["barcode1_only", "barcode2_only"]
                ).sum()
            )

            both_barcodes = int(
                (cell_df["classification"] == "both_barcodes").sum()
            )

            print()
            print("=" * 80)
            print("FINAL TWO NUMBERS — ALL SAMPLES COMBINED")
            print(f"  Cells with exactly one barcode: {one_barcode}")
            print(f"  Cells with both barcodes: {both_barcodes}")
            print("=" * 80)

            # ── FINAL TWO NUMBERS, EACH SAMPLE SEPARATELY ───────────────────
            print()
            print("=" * 80)
            print("FINAL TWO NUMBERS BY SAMPLE")
            print("=" * 80)

            summary_rows = []

            for sname, sub in cell_df.groupby("sample"):
                one_barcode_sample = int(
                    sub["classification"].isin(
                        ["barcode1_only", "barcode2_only"]
                    ).sum()
                )

                both_barcodes_sample = int(
                    (sub["classification"] == "both_barcodes").sum()
                )

                total_cells_sample = int(len(sub))

                no_barcode_sample = int(
                    (sub["classification"] == "no_barcode").sum()
                )

                barcode1_only_sample = int(
                    (sub["classification"] == "barcode1_only").sum()
                )

                barcode2_only_sample = int(
                    (sub["classification"] == "barcode2_only").sum()
                )

                summary_rows.append({
                    "sample": sname,
                    "total_cells": total_cells_sample,
                    "cells_with_exactly_one_barcode": one_barcode_sample,
                    "cells_with_both_barcodes": both_barcodes_sample,
                    "barcode1_only": barcode1_only_sample,
                    "barcode2_only": barcode2_only_sample,
                    "no_barcode": no_barcode_sample,
                })

                print(f"\n{sname}")
                print(f"  Total cells: {total_cells_sample}")
                print(f"  Cells with exactly one barcode: {one_barcode_sample}")
                print(f"  Cells with both barcodes: {both_barcodes_sample}")
                print(f"  Barcode1 only: {barcode1_only_sample}")
                print(f"  Barcode2 only: {barcode2_only_sample}")
                print(f"  No barcode: {no_barcode_sample}")

            summary_df = pd.DataFrame(summary_rows)

            summary_csv = OUTPUT_ROOT / "summary_2oligo_barcode_assignment_by_sample.csv"
            summary_df.to_csv(summary_csv, index=False)

            print()
            print(f"Saved summary table: {summary_csv}")

        else:
            print("WARNING: No per-cell records generated.")

else:
    print("Assignment skipped because RUN_ASSIGNMENT=False. Loading existing tables.")

    cell_df = pd.DataFrame()

    for sf in valid_samples:
        sf = Path(sf)
        csv_path = OUTPUT_ROOT / sf.name / "per_cell_2oligo_barcode_assignment.csv"

        if csv_path.exists():
            cell_df = pd.concat(
                [cell_df, pd.read_csv(csv_path)],
                ignore_index=True
            )
            print(f"Loaded: {csv_path}")
        else:
            print(f"No assignment CSV found for {sf.name}")

    print(f"\nLoaded {len(cell_df)} cell records.")

    if not cell_df.empty and EXCLUDE_FOVS_BY_SAMPLE:
        n_before_fov_qc = len(cell_df)
        keep_mask = pd.Series(True, index=cell_df.index)
        for sample_name, excluded_fovs in EXCLUDE_FOVS_BY_SAMPLE.items():
            keep_mask &= ~(
                (cell_df["sample"] == sample_name)
                & (cell_df["fov"].astype(int).isin(excluded_fovs))
            )
        cell_df = cell_df[keep_mask].copy()
        print(
            "Applying QC FOV exclusions to loaded cell table: "
            f"kept {len(cell_df)} / {n_before_fov_qc} cells"
        )

    if not cell_df.empty:
        print()
        print("Overall classification counts:")
        print(cell_df["classification"].value_counts().to_string())

        one_barcode = int(
            cell_df["classification"].isin(
                ["barcode1_only", "barcode2_only"]
            ).sum()
        )

        both_barcodes = int(
            (cell_df["classification"] == "both_barcodes").sum()
        )

        print()
        print("=" * 80)
        print("FINAL TWO NUMBERS — ALL SAMPLES COMBINED")
        print(f"  Cells with exactly one barcode: {one_barcode}")
        print(f"  Cells with both barcodes: {both_barcodes}")
        print("=" * 80)

        print()
        print("=" * 80)
        print("FINAL TWO NUMBERS BY SAMPLE")
        print("=" * 80)

        summary_rows = []

        for sname, sub in cell_df.groupby("sample"):
            one_barcode_sample = int(
                sub["classification"].isin(
                    ["barcode1_only", "barcode2_only"]
                ).sum()
            )

            both_barcodes_sample = int(
                (sub["classification"] == "both_barcodes").sum()
            )

            total_cells_sample = int(len(sub))

            no_barcode_sample = int(
                (sub["classification"] == "no_barcode").sum()
            )

            barcode1_only_sample = int(
                (sub["classification"] == "barcode1_only").sum()
            )

            barcode2_only_sample = int(
                (sub["classification"] == "barcode2_only").sum()
            )

            summary_rows.append({
                "sample": sname,
                "total_cells": total_cells_sample,
                "cells_with_exactly_one_barcode": one_barcode_sample,
                "cells_with_both_barcodes": both_barcodes_sample,
                "barcode1_only": barcode1_only_sample,
                "barcode2_only": barcode2_only_sample,
                "no_barcode": no_barcode_sample,
            })

            print(f"\n{sname}")
            print(f"  Total cells: {total_cells_sample}")
            print(f"  Cells with exactly one barcode: {one_barcode_sample}")
            print(f"  Cells with both barcodes: {both_barcodes_sample}")
            print(f"  Barcode1 only: {barcode1_only_sample}")
            print(f"  Barcode2 only: {barcode2_only_sample}")
            print(f"  No barcode: {no_barcode_sample}")

        summary_df = pd.DataFrame(summary_rows)

        summary_csv = OUTPUT_ROOT / "summary_2oligo_barcode_assignment_by_sample.csv"
        summary_df.to_csv(summary_csv, index=False)

        print()
        print(f"Saved summary table: {summary_csv}")


## Final Quantification

In [ ]:
from pathlib import Path
import pandas as pd

def _pct(n, total):
    return f"{100*n/total:.1f}%" if total > 0 else "N/A"


if cell_df.empty:
    print("No cell data. Run spot detection, segmentation, and assignment first.")

else:
    summary_rows = []

    # Loop through each sample separately
    for sname in sorted(cell_df["sample"].unique()):
        sub = cell_df[cell_df["sample"] == sname].copy()

        n = len(sub)

        n_b1 = int((sub["classification"] == "barcode1_only").sum())
        n_b2 = int((sub["classification"] == "barcode2_only").sum())
        n_both = int((sub["classification"] == "both_barcodes").sum())
        n_none = int((sub["classification"] == "no_barcode").sum())

        n_one = n_b1 + n_b2

        print("=" * 60)
        print(f"Sample: {sname}")
        print("=" * 60)
        print(f"  Total segmented cells:      {n}")
        print(f"  Barcode 1 only cells:       {n_b1:6d}  ({_pct(n_b1, n)})")
        print(f"  Barcode 2 only cells:       {n_b2:6d}  ({_pct(n_b2, n)})")
        print(f"  Exactly one barcode cells:  {n_one:6d}  ({_pct(n_one, n)})")
        print(f"  Both barcode cells:         {n_both:6d}  ({_pct(n_both, n)})")
        print(f"  No barcode cells:           {n_none:6d}  ({_pct(n_none, n)})")
        print()

        summary_rows.append({
            "sample": sname,
            "total_cells": n,

            "barcode1_only_count": n_b1,
            "barcode1_only_pct": round(100 * n_b1 / n, 2) if n else 0,

            "barcode2_only_count": n_b2,
            "barcode2_only_pct": round(100 * n_b2 / n, 2) if n else 0,

            "one_barcode_count": n_one,
            "one_barcode_pct": round(100 * n_one / n, 2) if n else 0,

            "both_barcodes_count": n_both,
            "both_barcodes_pct": round(100 * n_both / n, 2) if n else 0,

            "no_barcode_count": n_none,
            "no_barcode_pct": round(100 * n_none / n, 2) if n else 0,
        })

    summary_df = pd.DataFrame(summary_rows)

    # ── Save one final summary CSV inside each sample folder ────────────────
    for sname in summary_df["sample"]:
        sample_out = OUTPUT_ROOT / sname
        sample_out.mkdir(parents=True, exist_ok=True)

        csv_path = sample_out / "final_2oligo_cell_counts.csv"

        summary_df[summary_df["sample"] == sname].to_csv(
            csv_path,
            index=False
        )

        print(f"Saved individual sample summary: {csv_path}")

    # ── Save one combined summary CSV across all samples ────────────────────
    combined = OUTPUT_ROOT / "final_2oligo_cell_counts_all_samples.csv"

    summary_df.to_csv(
        combined,
        index=False
    )

    print()
    print(f"Saved combined summary: {combined}")

    print()
    print("Final summary table:")
    display(summary_df)

### Archived CSV outputs

The original run's spot tables, per-cell assignments, per-sample summaries, combined summaries, and statistical tables are archived under `../Data/Supplementary_Figure_1c_2Oligo_RCA/Processed_Data/`. The cells above show how each table is constructed; runtime copies are temporary.


## Visualization/QC

Overlays segmentation boundaries, barcode1 spots (yellow circles), and barcode2 spots (cyan triangles) on the DAPI image for a few FOVs per sample. Saves PNG overlays to `OUTPUT_ROOT/qc_visualizations/`.

In [ ]:
if RUN_VISUALIZATION and mask_results:

    def _norm(img, p_lo=1, p_hi=99.5):
        vmin, vmax = np.percentile(img, [p_lo, p_hi])
        if vmax <= vmin:
            vmax = vmin + 1
        return np.clip((img - vmin) / (vmax - vmin), 0.0, 1.0)

    for sf in valid_samples[:N_VIS_SAMPLES]:
        sname    = sf.name
        img_dir  = sf / "0"

        tile_m = re.match(r'^(.+?)_\d+_\d+_', sorted(img_dir.glob("*.tiff"))[0].name)
        tile_prefix = tile_m.group(1) if tile_m else None
        if tile_prefix is None:
            continue

        sample_fovs = sorted([f for (s, f) in mask_results if s == sname])
        vis_fovs    = sample_fovs[:N_VIS_FOVS]

        for fov_id in vis_fovs:
            masks = mask_results[(sname, fov_id)]

            dapi_path = img_dir / f"{tile_prefix}_{fov_id}_0_{CHANNEL_FILE_SUFFIX['DAPI']}.tiff"
            b1_path   = img_dir / f"{tile_prefix}_{fov_id}_0_{CHANNEL_FILE_SUFFIX['Cy3-F46']}.tiff"
            b2_path   = img_dir / f"{tile_prefix}_{fov_id}_0_{CHANNEL_FILE_SUFFIX['Cy5-F20']}.tiff"

            if not all(p.exists() for p in [dapi_path, b1_path, b2_path]):
                print(f"  Skipping FOV {fov_id}: missing image file(s).")
                continue

            dapi_n = _norm(read_yx(dapi_path))
            b1_n   = _norm(read_yx(b1_path))
            b2_n   = _norm(read_yx(b2_path))

            # Get spots for this FOV
            if not spots_df.empty:
                fov_spots = spots_df[(spots_df["sample"]==sname) & (spots_df["fov"]==fov_id)]
                b1_sp = fov_spots[fov_spots["barcode"]=="barcode1"]
                b2_sp = fov_spots[fov_spots["barcode"]=="barcode2"]
            else:
                b1_sp = b2_sp = pd.DataFrame(columns=["x","y"])

            fig, axes = plt.subplots(1, 3, figsize=(18, 6))

            # Panel 1: DAPI + nucleus boundaries
            axes[0].imshow(dapi_n, cmap="gray", vmin=0, vmax=1)
            axes[0].contour(masks, levels=[0.5], colors="cyan", linewidths=0.5)
            axes[0].set_title(f"DAPI + nuclei  ({int(masks.max())} cells)")
            axes[0].axis("off")

            # Panel 2: barcode1 channel + spots
            axes[1].imshow(b1_n, cmap="hot", vmin=0, vmax=1)
            if not b1_sp.empty:
                axes[1].scatter(b1_sp["x"], b1_sp["y"],
                                s=12, c="yellow", marker="o", alpha=0.8,
                                linewidths=0.4, edgecolors="black",
                                label=f"B1 spots  n={len(b1_sp)}")
                axes[1].legend(loc="upper right", fontsize=8)
            axes[1].set_title(f"Barcode 1  Cy3-F46  ({len(b1_sp)} spots)")
            axes[1].axis("off")

            # Panel 3: barcode2 channel + spots
            axes[2].imshow(b2_n, cmap="Blues", vmin=0, vmax=1)
            if not b2_sp.empty:
                axes[2].scatter(b2_sp["x"], b2_sp["y"],
                                s=12, c="cyan", marker="^", alpha=0.8,
                                linewidths=0.4, edgecolors="white",
                                label=f"B2 spots  n={len(b2_sp)}")
                axes[2].legend(loc="upper right", fontsize=8)
            axes[2].set_title(f"Barcode 2  Cy5-F20  ({len(b2_sp)} spots)")
            axes[2].axis("off")

            plt.suptitle(f"{sname}  –  FOV {fov_id}", fontsize=13, y=1.01)
            plt.tight_layout()

            plt.show()
            plt.close(fig)

    print("\nQC comparison displayed above; no image file was exported.")

elif not RUN_VISUALIZATION:
    print("Visualization skipped (RUN_VISUALIZATION=False).")
else:
    print("No segmentation masks available for visualization.")


## Well Group Assignment and RCA Bar Plot

Assigns well-prefixed samples to the requested treatment groups, leaves missing or non-well-prefixed samples unassigned, and plots grouped bar graphs with individual wells overlaid.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, ttest_ind

# Requested plate map:
# B2-6 PBS without RCA, B7-11 PBS with RCA
# C2-6 0.1ug/0.1ug without RCA, C7-11 0.1ug/0.1ug with RCA
# D2-6 0.01ug/0.01ug without RCA, D7-11 0.01ug/0.01ug with RCA
# E2-6 0.001ug/0.001ug without RCA, E7-11 0.001ug/0.001ug with RCA
WELL_GROUP_MAP = {
    "B": {"formulation": "PBS", "dose_order": 0},
    "C": {"formulation": "100ng", "dose_order": 1},
    "D": {"formulation": "10ng", "dose_order": 2},
    "E": {"formulation": "1ng", "dose_order": 3},
}

RCA_STATUS_BY_COLUMN = {
    "without RCA": range(2, 7),
    "with RCA": range(7, 12),
}

# Change this if you want to plot another endpoint from summary_df.
# Good options: "both_barcodes_pct", "one_barcode_pct", "both_barcodes_count", "one_barcode_count".
GROUP_PLOT_VALUE_COL = "both_barcodes_pct"
GROUP_PLOT_YLABEL = "% cells with both barcodes"
SHOW_FOV_TECHNICAL_REPLICATES = True  # show FOVs as technical replicate dots; bars stay well-level
MIN_CELLS_PER_FOV_FOR_TECHNICAL_REPLICATES = 10  # skip non-cell / near-empty FOVs

# Exclude the original C3 run but keep C3_1.
# Also exclude D1-D6, which are the 10ng without-RCA wells.
EXCLUDE_SAMPLE_PATTERNS_FROM_GROUP_PLOT = [
    r"^C3_(?!1(?:_|$))",
    r"^D[1-6](?:_|$)",
]


def _extract_well(sample_name):
    """Return B10/C3/etc. from sample names such as B10_..., C3_1_..., or None."""
    m = re.match(r"^([A-Ha-h])(\d{1,2})(?:_|$)", str(sample_name))
    if not m:
        return None
    return f"{m.group(1).upper()}{int(m.group(2))}"


def _p_to_stars(p_value):
    if pd.isna(p_value):
        return "NS"
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "NS"


def _format_p_value(p_value):
    if pd.isna(p_value):
        return "P = NA"
    if p_value < 0.0001:
        return "P < 0.0001"
    return f"P = {p_value:.4f}"


def _add_pairwise_pvalue(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color="black", linewidth=1)
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom", fontsize=10)


def _assign_well_group(sample_name):
    well = _extract_well(sample_name)
    if well is None:
        if str(sample_name).upper().startswith("PBS"):
            return {
                "well": "PBS",
                "formulation": "PBS",
                "rca_status": "without RCA",
                "group_label": "PBS, without RCA",
                "dose_order": 0,
                "rca_order": 0,
            }
        return {
            "well": None,
            "formulation": None,
            "rca_status": None,
            "group_label": None,
            "dose_order": np.nan,
            "rca_order": np.nan,
        }

    row = well[0]
    col = int(well[1:])
    base = WELL_GROUP_MAP.get(row)
    if base is None:
        rca_status = None
        rca_order = np.nan
    else:
        rca_status = None
        rca_order = np.nan
        for status, cols in RCA_STATUS_BY_COLUMN.items():
            if col in cols:
                rca_status = status
                rca_order = 0 if status == "without RCA" else 1
                break

    if base is None or rca_status is None:
        formulation = None
        group_label = None
        dose_order = np.nan
    else:
        formulation = base["formulation"]
        dose_order = base["dose_order"]
        group_label = f"{formulation}, {rca_status}"

    return {
        "well": well,
        "formulation": formulation,
        "rca_status": rca_status,
        "group_label": group_label,
        "dose_order": dose_order,
        "rca_order": rca_order,
    }


# Use the in-memory summary_df when available; otherwise load the saved summary.
try:
    _summary_source = summary_df.copy()
except NameError:
    summary_path = OUTPUT_ROOT / "final_2oligo_cell_counts_all_samples.csv"
    _summary_source = pd.read_csv(summary_path)

if _summary_source.empty:
    print("No summary data available. Run the quantification cells first.")
elif GROUP_PLOT_VALUE_COL not in _summary_source.columns:
    print(f"Column not found: {GROUP_PLOT_VALUE_COL}")
    print("Available columns:", list(_summary_source.columns))
else:
    plots_dir = OUTPUT_ROOT / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    if EXCLUDE_SAMPLE_PATTERNS_FROM_GROUP_PLOT:
        exclude_pattern = "|".join(EXCLUDE_SAMPLE_PATTERNS_FROM_GROUP_PLOT)
        exclude_mask = _summary_source["sample"].astype(str).str.contains(exclude_pattern, regex=True)
        excluded_samples = _summary_source.loc[exclude_mask, "sample"].tolist()
        _summary_source = _summary_source.loc[~exclude_mask].copy()
        if excluded_samples:
            print("Excluded samples from grouped plot:", excluded_samples)

    group_info = _summary_source["sample"].apply(_assign_well_group).apply(pd.Series)
    grouped_well_df = pd.concat([_summary_source.copy(), group_info], axis=1)

    assigned_df = grouped_well_df.dropna(subset=["group_label"]).copy()
    unassigned_df = grouped_well_df[grouped_well_df["group_label"].isna()].copy()

    grouped_csv = OUTPUT_ROOT / "well_grouped_2oligo_summary.csv"
    grouped_well_df.to_csv(grouped_csv, index=False)
    print(f"Saved well-grouped summary: {grouped_csv}")

    if not unassigned_df.empty:
        print("\nUnassigned samples, left out of grouped bar plot:")
        display(unassigned_df[["sample", "well"]])

    if assigned_df.empty:
        print("No assigned well-prefixed samples to plot.")
    else:
        assigned_df[GROUP_PLOT_VALUE_COL] = pd.to_numeric(
            assigned_df[GROUP_PLOT_VALUE_COL], errors="coerce"
        )
        assigned_df = assigned_df.dropna(subset=[GROUP_PLOT_VALUE_COL]).copy()
        assigned_df = assigned_df.sort_values(["dose_order", "rca_order", "well", "sample"])

        stats_df = (
            assigned_df
            .groupby(["formulation", "rca_status", "dose_order", "rca_order"], dropna=False)
            .agg(
                n=(GROUP_PLOT_VALUE_COL, "count"),
                mean=(GROUP_PLOT_VALUE_COL, "mean"),
                sd=(GROUP_PLOT_VALUE_COL, "std"),
            )
            .reset_index()
        )
        stats_df["sem"] = stats_df["sd"] / np.sqrt(stats_df["n"])
        stats_df["sem"] = stats_df["sem"].fillna(0)
        stats_df = stats_df.sort_values(["dose_order", "rca_order"])

        stats_csv = OUTPUT_ROOT / "well_grouped_2oligo_stats.csv"
        stats_df.to_csv(stats_csv, index=False)
        print(f"Saved grouped stats: {stats_csv}")
        display(stats_df)

        fov_plot_df = pd.DataFrame()
        if SHOW_FOV_TECHNICAL_REPLICATES:
            cell_csv = OUTPUT_ROOT / "combined_per_cell_2oligo_barcode_assignment.csv"
            if cell_csv.exists():
                cell_for_fov_df = pd.read_csv(cell_csv)

                # Respect the same sample and FOV QC exclusions as the well-level plot.
                if EXCLUDE_SAMPLE_PATTERNS_FROM_GROUP_PLOT:
                    exclude_pattern = "|".join(EXCLUDE_SAMPLE_PATTERNS_FROM_GROUP_PLOT)
                    keep_samples = ~cell_for_fov_df["sample"].astype(str).str.contains(exclude_pattern, regex=True)
                    cell_for_fov_df = cell_for_fov_df.loc[keep_samples].copy()

                fov_exclusions = globals().get("EXCLUDE_FOVS_BY_SAMPLE", {})
                if fov_exclusions:
                    keep_fovs = pd.Series(True, index=cell_for_fov_df.index)
                    for sample_name, excluded_fovs in fov_exclusions.items():
                        keep_fovs &= ~(
                            (cell_for_fov_df["sample"] == sample_name)
                            & (cell_for_fov_df["fov"].astype(int).isin(excluded_fovs))
                        )
                    cell_for_fov_df = cell_for_fov_df.loc[keep_fovs].copy()

                fov_rows = []
                for (sample_name, fov_id), sub in cell_for_fov_df.groupby(["sample", "fov"]):
                    group_info = _assign_well_group(sample_name)
                    if group_info["group_label"] is None:
                        continue

                    n_cells = len(sub)
                    if n_cells < MIN_CELLS_PER_FOV_FOR_TECHNICAL_REPLICATES:
                        continue

                    n_b1 = int((sub["classification"] == "barcode1_only").sum())
                    n_b2 = int((sub["classification"] == "barcode2_only").sum())
                    n_both = int((sub["classification"] == "both_barcodes").sum())
                    n_one = n_b1 + n_b2

                    value_lookup = {
                        "both_barcodes_pct": 100 * n_both / n_cells if n_cells else np.nan,
                        "one_barcode_pct": 100 * n_one / n_cells if n_cells else np.nan,
                        "both_barcodes_count": n_both,
                        "one_barcode_count": n_one,
                    }
                    if GROUP_PLOT_VALUE_COL not in value_lookup:
                        continue

                    fov_rows.append({
                        "sample": sample_name,
                        "fov": int(fov_id),
                        "n_cells": n_cells,
                        "value": value_lookup[GROUP_PLOT_VALUE_COL],
                        **group_info,
                    })

                fov_plot_df = pd.DataFrame(fov_rows)
                if not fov_plot_df.empty:
                    fov_csv = OUTPUT_ROOT / f"well_grouped_fov_technical_replicates_{GROUP_PLOT_VALUE_COL}.csv"
                    fov_plot_df.to_csv(fov_csv, index=False)
                    print(f"Saved FOV technical replicate table: {fov_csv}")
            else:
                print(f"No combined cell table found for FOV dots: {cell_csv}")

        formulations = [
            WELL_GROUP_MAP[row]["formulation"]
            for row in ["B", "C", "D", "E"]
        ]
        status_order = ["without RCA", "with RCA"]
        colors = {"without RCA": "#8a8a8a", "with RCA": "#ffd9ad"}
        point_colors = {"without RCA": "#000000", "with RCA": "#ff6b1a"}

        x = np.arange(len(formulations), dtype=float)
        width = 0.34
        offsets = {"without RCA": -width / 2, "with RCA": width / 2}

        fig, ax = plt.subplots(figsize=(7.2, 4.8))

        ymax = 0
        for status in status_order:
            means = []
            sems = []
            for formulation in formulations:
                row = stats_df[
                    (stats_df["formulation"] == formulation)
                    & (stats_df["rca_status"] == status)
                ]
                if row.empty:
                    means.append(np.nan)
                    sems.append(0)
                else:
                    means.append(float(row["mean"].iloc[0]))
                    sems.append(float(row["sem"].iloc[0]))

            bar_x = x + offsets[status]
            ax.bar(
                bar_x,
                means,
                width=width,
                yerr=sems,
                capsize=4,
                color=colors[status],
                edgecolor="none",
                linewidth=0,
                label=status,
                zorder=2,
            )

            for i, formulation in enumerate(formulations):
                if SHOW_FOV_TECHNICAL_REPLICATES and not fov_plot_df.empty:
                    fov_vals = fov_plot_df.loc[
                        (fov_plot_df["formulation"] == formulation)
                        & (fov_plot_df["rca_status"] == status),
                        "value",
                    ].astype(float).to_numpy()
                    if len(fov_vals) > 0:
                        jitter = np.linspace(-0.11, 0.11, len(fov_vals)) if len(fov_vals) > 1 else np.array([0.0])
                        ax.scatter(
                            np.repeat(bar_x[i], len(fov_vals)) + jitter,
                            fov_vals,
                            s=13,
                            color=point_colors[status],
                            alpha=0.28,
                            edgecolor="none",
                            zorder=3,
                        )
                        ymax = max(ymax, np.nanmax(fov_vals))

                vals = assigned_df.loc[
                    (assigned_df["formulation"] == formulation)
                    & (assigned_df["rca_status"] == status),
                    GROUP_PLOT_VALUE_COL,
                ].astype(float).to_numpy()
                if len(vals) == 0:
                    continue
                jitter = np.linspace(-0.045, 0.045, len(vals)) if len(vals) > 1 else np.array([0.0])
                ax.scatter(
                    np.repeat(bar_x[i], len(vals)) + jitter,
                    vals,
                    s=38,
                    color=point_colors[status],
                    edgecolor="white",
                    linewidth=0.5,
                    zorder=4,
                )
                ymax = max(ymax, np.nanmax(vals), np.nanmax(means) if len(means) else 0)

        pvalue_ymax = ymax
        pvalue_formulation = "100ng"
        if SHOW_FOV_TECHNICAL_REPLICATES and not fov_plot_df.empty:
            pvalue_without = fov_plot_df.loc[
                (fov_plot_df["formulation"] == pvalue_formulation)
                & (fov_plot_df["rca_status"] == "without RCA"),
                "value",
            ].astype(float).to_numpy()
            pvalue_with = fov_plot_df.loc[
                (fov_plot_df["formulation"] == pvalue_formulation)
                & (fov_plot_df["rca_status"] == "with RCA"),
                "value",
            ].astype(float).to_numpy()
            stat_label = "FOV-level Welch t-test"
        else:
            pvalue_without = np.array([])
            pvalue_with = np.array([])
            stat_label = "FOV-level Welch t-test"

        if len(pvalue_without) >= 2 and len(pvalue_with) >= 2:
            p_value = float(ttest_ind(pvalue_without, pvalue_with, equal_var=False).pvalue)
            i = formulations.index(pvalue_formulation)
            x1 = x[i] + offsets["without RCA"]
            x2 = x[i] + offsets["with RCA"]

            well_without = assigned_df.loc[
                (assigned_df["formulation"] == pvalue_formulation)
                & (assigned_df["rca_status"] == "without RCA"),
                GROUP_PLOT_VALUE_COL,
            ].astype(float).to_numpy()
            well_with = assigned_df.loc[
                (assigned_df["formulation"] == pvalue_formulation)
                & (assigned_df["rca_status"] == "with RCA"),
                GROUP_PLOT_VALUE_COL,
            ].astype(float).to_numpy()
            local_arrays = [
                a for a in [pvalue_without, pvalue_with, well_without, well_with]
                if len(a) > 0
            ]
            local_ymax = np.nanmax(np.concatenate(local_arrays))
            h = max(local_ymax * 0.06, 0.6)
            y = local_ymax + h
            label = f"{_p_to_stars(p_value)} {_format_p_value(p_value)}"
            _add_pairwise_pvalue(ax, x1, x2, y, h, label)
            pvalue_ymax = max(pvalue_ymax, y + h)
            print(
                f"{stat_label} {pvalue_formulation}, with vs without RCA: "
                f"p = {p_value:.6g}; n_without={len(pvalue_without)}, n_with={len(pvalue_with)}"
            )

            fov_stat_csv = OUTPUT_ROOT / f"well_grouped_{pvalue_formulation}_fov_level_stats.csv"
            pd.DataFrame([{
                "formulation": pvalue_formulation,
                "stat_level": "FOV",
                "test": "Welch two-sample t-test",
                "n_without_rca_fovs": len(pvalue_without),
                "n_with_rca_fovs": len(pvalue_with),
                "without_rca_mean": np.nanmean(pvalue_without),
                "with_rca_mean": np.nanmean(pvalue_with),
                "p_value": p_value,
                "stars": _p_to_stars(p_value),
            }]).to_csv(fov_stat_csv, index=False)
            print(f"Saved FOV-level 100ng statistics: {fov_stat_csv}")
        else:
            print(
                f"Skipping FOV-level t-test for {pvalue_formulation}: "
                f"need n >= 2 in each group, got without={len(pvalue_without)}, with={len(pvalue_with)}"
            )

        ax.set_ylabel(GROUP_PLOT_YLABEL, fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(formulations, rotation=35, ha="right", fontsize=10)
        ax.set_xlabel("Treatment", fontsize=12)
        ax.set_title("2-Oligo quantification by well group", fontsize=13, pad=10)
        ax.legend(frameon=False, fontsize=10)
        if SHOW_FOV_TECHNICAL_REPLICATES and not fov_plot_df.empty:
            ax.text(
                0.99,
                0.98,
                "large dots: wells\nsmall faint dots: FOVs",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=8,
                color="#555555",
            )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", labelsize=10)
        ax.set_ylim(0, max(5, pvalue_ymax * 1.25))

        plt.tight_layout()

        plt.show()
        plt.close(fig)

        print("Supplementary Figure 1c displayed; no figure file was exported.")












## Reference output from the original run

This archived image is displayed only for visual comparison with the in-notebook plot above.

![Supplementary Figure 1c reference](../Data/Supplementary_Figure_1c_2Oligo_RCA/Processed_Data/plots/well_grouped_2oligo_quantification_nature.png)


In [ ]:
# Remove all temporary stitching images, masks, and runtime CSVs.
_runtime_cache.cleanup()
print("Temporary runtime files removed. No persistent outputs were written.")
